In [16]:
import pandas as pd
import altair as alt


data = 'https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/licenses_fall2022.csv'
df = pd.read_csv(data)

In [17]:
df

,_id,License Type,Description,License Number,License Status,Business,Title,First Name,Middle,Last Name,...,Specialty/Qualifier,Controlled Substance Schedule,Delegated Controlled Substance Schedule,Ever Disciplined,LastModifiedDate,Case Number,Action,Discipline Start Date,Discipline End Date,Discipline Reason
0,1189509,DETECTIVE BOARD,PERMANENT EMPLOYEE REGISTRATION,129446286,NOT RENEWED,N,NaN,EILEEN,NaN,SANTACRUZ,...,NaN,NaN,NaN,N,03/18/2022,NaN,NaN,NaN,NaN,NaN
1,801037,DETECTIVE BOARD,FIREARM CONTROL CARD,229030294.0,NOT RENEWED,N,NaN,DAGMAR,J,NORDLUND,...,NaN,NaN,NaN,N,08/16/2006,NaN,NaN,NaN,NaN,NaN
2,365129,COSMO,LICENSED COSMETOLOGIST,11053076.0,NOT RENEWED,N,NaN,RADOJE,NaN,ZELENOVIC,...,NaN,NaN,NaN,N,05/26/2006,NaN,NaN,NaN,NaN,NaN
3,595427,COSMO,LICENSED COSMETOLOGIST,11295645.0,ACTIVE,N,NaN,BECKY SUE,L,BURROUGHS,...,NaN,NaN,NaN,N,11/12/2021,NaN,NaN,NaN,NaN,NaN
4,653668,COSMO,LICENSED NAIL TECHNICIAN,169006247,NOT RENEWED,N,NaN,BILL G,L,LETNER,...,NaN,NaN,NaN,N,05/30/2006,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,888281,DETECTIVE BOARD,PERMANENT EMPLOYEE REGISTRATION,129002843.0,NOT RENEWED,N,NaN,JENNIFER,NaN,DARROW,...,NaN,NaN,NaN,N,08/03/2006,NaN,NaN,NaN,NaN,NaN
9996,766623,DETECTIVE BOARD,FIREARM CONTROL CARD,229014180,TERMINATED CARD RETURNED,N,NaN,BRYAN,NaN,WILLIAMS,...,NaN,NaN,NaN,N,08/07/2006,NaN,NaN,NaN,NaN,NaN
9997,399398,COSMO,LICENSED COSMETOLOGIST,11120249,NOT RENEWED,N,NaN,EUGENE,NaN,HENDERSON JR,...,NaN,NaN,NaN,N,05/26/2006,NaN,NaN,NaN,NaN,NaN
9998,486713,COSMO,LICENSED COSMETOLOGIST,11193270,ACTIVE,N,NaN,MAHLON DOUGLAS,NaN,CLIFT,...,NaN,NaN,NaN,N,12/17/2021,NaN,NaN,NaN,NaN,NaN


In [18]:
df.dtypes

_id                                          int64
License Type                                object
Description                                 object
License Number                              object
License Status                              object
Business                                    object
Title                                       object
First Name                                  object
Middle                                      object
Last Name                                   object
Prefix                                      object
Suffix                                      object
Business Name                               object
BusinessDBA                                 object
Original Issue Date                         object
Effective Date                              object
Expiration Date                             object
City                                        object
State                                       object
Zip                            

In [19]:
df.isna().sum()

_id                                            0
License Type                                   0
Description                                    0
License Number                                60
License Status                                 0
Business                                       0
Title                                       9890
First Name                                   395
Middle                                      6378
Last Name                                    395
Prefix                                      9997
Suffix                                      9590
Business Name                                  0
BusinessDBA                                 9885
Original Issue Date                            5
Effective Date                               792
Expiration Date                              500
City                                          11
State                                          0
Zip                                           71
County              

In [20]:
df['License Type'].value_counts()

License Type
DETECTIVE BOARD           4867
COSMO                     3781
DENTAL                     739
FUNERAL AND EMBALMER        98
DIETETIC AND NUTRITION      73
DESIGN FIRM                 71
MASSAGE LICENSING BD        52
HOME INSPECTOR              46
COMM ASSOC MGR              37
CLIN PSYCHOLOGIST           24
INTERIOR DESIGN             22
COLLECTION AGENCY           21
LIMITED LIABILITY CO        21
ARCHITECT                   20
LAND SURVEYOR BOARD         17
HME AND SERVICES PROV       16
ATHLETICS                   15
MAR AND FAM THERAPIST       14
ATHLETIC TRAINER            14
ENVIRON. HLTH PRACT         13
IDPR                         9
GEOLOGY                      7
LANDSCAPE ARCHITECT          7
MEDICAL BOARD                6
APPRAISAL                    5
DETECT. DECEPTION            2
AUCTIONEER                   2
CEMETERY OVERSIGHT           1
Name: count, dtype: int64

## Plot 1: Top License Types by Count - interactive

In [21]:
# Count licenses per License Type + keep the top 20
top_n = 20
type_counts = (
    df['License Type']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'License Type', 'License Type': 'Count'})
    .head(top_n)
)
type_counts.columns = ['License Type', 'Count']

In [22]:
type_counts.head()

,License Type,Count
0,DETECTIVE BOARD,4867
1,COSMO,3781
2,DENTAL,739
3,FUNERAL AND EMBALMER,98
4,DIETETIC AND NUTRITION,73


In [23]:
selection = alt.selection_point(fields=['License Type'])

color_condition = alt.condition(
    selection,
    alt.Color('Count:Q',
              scale=alt.Scale(scheme='tealblues'),
              legend=None),
    alt.value('lightgray')
)

plot1 = alt.Chart(type_counts).mark_bar().encode(
    x=alt.X('Count:Q', title='Number of Licenses'),
    y=alt.Y('License Type:N',
            sort='-x',
            title='License Type'),
    color=color_condition,
    tooltip=[
        alt.Tooltip('License Type:N', title='License Type'),
        alt.Tooltip('Count:Q', title='# Licenses', format=',')
    ]
).add_params(
    selection
).properties(
    width=550,
    height=450,
    title='Top 20 Illinois Professional License Types'
)

plot1

alt.Chart(...)

In [24]:
plot1.save('plot1.json')

## Plot 2: License Status Breakdown by Top License Types 

In [25]:
# keeping only top 10 license types
top10_types = type_counts['License Type'].head(10).tolist()

df_top10 = df[df['License Type'].isin(top10_types)].copy()

In [26]:
# Group by License Type + License Status
status_counts = (df_top10.groupby(['License Type', 'License Status']).size().reset_index(name='Count'))

In [27]:
# Keep only the most common statuses to avoid messy plots
top_statuses = (status_counts.groupby('License Status')['Count'].sum().nlargest(5).index.tolist()
)
status_counts = status_counts[status_counts['License Status'].isin(top_statuses)]

In [15]:

plot2 = alt.Chart(status_counts).mark_bar().encode(
    y=alt.Y('License Type:N',
            sort=alt.EncodingSortField(field='Count', op='sum', order='descending'),
            title='License Type'),
    x=alt.X('Count:Q',
             stack='normalize',
             title='Proportion of Licenses',
             axis=alt.Axis(format='%')),
    color=alt.Color('License Status:N',
                    scale=alt.Scale(scheme='tableau10'),
                    title='License Status')
).properties(
    width=550,
    height=400,
    title='License Status Breakdown for Top 10 License Types (Normalized)'
)

plot2

alt.Chart(...)

In [14]:
plot2.save('plot2.json')